# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 3 — Salidas estructuradas y aplicaciones robustas (OpenAI)

ITBA Educación Ejecutiva

Generar resultados que otros sistemas puedan consumir de forma confiable — no texto libre que hay que parsear a mano y cruzar los dedos.

> Variante de `clase-03.ipynb` con **OpenAI** como proveedor de la demo en vivo (en lugar de AWS Bedrock). Los conceptos y el laboratorio son los mismos.

### Cómo se usa este notebook

- **Durante la clase en vivo**: el docente corre y explica cada celda (demo con OpenAI). Los alumnos no programan en simultáneo.
- **Celdas 🏠 Practicá**: tarea individual para después de la clase.
- **Sección 9 (Laboratorio)**: tarea principal de la semana — checkpoint del proyecto integrador. Se resuelve con el proveedor de cada uno (Bedrock, Anthropic, OpenAI, Google u Ollama), no hace falta que sea OpenAI.

## Objetivo de la clase

Al final de la clase vas a poder:
- Definir un JSON Schema y un modelo Pydantic equivalente para validar salidas de un LLM.
- Obtener salidas estructuradas usando el mecanismo nativo del proveedor cuando existe, y un patrón de reintento universal cuando no.
- Usar tool calling para que el modelo pida ejecutar una función externa con argumentos validados.
- Configurar timeouts, reintentos con backoff+jitter, y registrar tokens/costo/latencia de cada llamada.
- Consumir una respuesta en streaming.
- Refactorizar el cliente de la Clase 2 detrás de una interfaz común (`LLMProvider`), agnóstica de proveedor.

**Checkpoint del proyecto integrador de hoy**: servicio que devuelve salidas estructuradas y validadas (Pydantic) a partir de texto libre, sobre el cliente ya agnóstico de proveedor (ver `proyecto-integrador/README.md`, checkpoint Clase 3).

## Agenda

1. JSON Schema y Pydantic
2. Structured outputs (mecanismo nativo del proveedor)
3. Manejo de respuestas inválidas (patrón universal de reintento)
4. Tool calling
5. Timeouts, reintentos y backoff
6. Observabilidad básica: tokens, costo y latencia
7. Streaming
8. Abstracción de proveedores
9. Laboratorio: servicio de extracción estructurada

## 0. Punto de partida

Recordá las funciones `llamar_bedrock`, `llamar_anthropic`, `llamar_openai`, `llamar_google` y `llamar_ollama` de la Clase 2 — hoy las vamos a envolver en clases con una interfaz común. Para que este notebook sea autocontenido, volvemos a definir `llamar_openai` acá (es la misma que ya tenías).

In [ ]:
import os
import json
import time
import random
from dotenv import load_dotenv

load_dotenv()
provider = os.environ.get("LLM_PROVIDER", "openai")
print(f"Proveedor configurado: {provider}")
assert os.environ.get("OPENAI_API_KEY"), (
    "Falta OPENAI_API_KEY en el .env — copiá .env.example a .env y completá la sección de OpenAI."
)

In [ ]:
provider

In [ ]:
# --- Función de la Clase 2 (recordatorio, variante OpenAI) ---
import openai
from openai import OpenAI

def llamar_openai(prompt, system=None, temperature=0.7, max_tokens=500):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    try:
        resp = client.chat.completions.create(
            model=os.environ.get("OPENAI_MODEL_ID", "gpt-4o"),
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content
    except openai.RateLimitError:
        raise
    except openai.APIStatusError as e:
        raise RuntimeError(f"Error de la API de OpenAI: {e.status_code}") from e

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
model_id = os.environ.get("OPENAI_MODEL_ID", "gpt-4o")
print(f"Modelo: {model_id}")

## 1. JSON Schema y Pydantic

- **JSON Schema**: un documento (también JSON) que describe la forma esperada de otro documento JSON — tipos, campos requeridos, enums, etc. Es el estándar que entienden las APIs de los proveedores para "forzar" un formato de salida.
- **Pydantic**: librería de Python que define esa misma forma como una clase, valida datos contra ella, y puede generar el JSON Schema automáticamente — no hace falta escribirlo dos veces.

In [ ]:
# %pip install pydantic  # si hace falta instalarlo

from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class ConsultaSoporte(BaseModel):
    categoria: Literal["facturacion", "tecnico", "cuenta", "otro"]
    urgencia: Literal["baja", "media", "alta"]
    resumen: str = Field(..., max_length=200, description="Resumen breve del reclamo, en una oración")


# El JSON Schema se genera solo a partir de la clase:
schema = ConsultaSoporte.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))

In [ ]:
# Validación: un dict válido pasa, uno inválido lanza ValidationError con el detalle exacto de qué falló
dato_valido = {"categoria": "tecnico", "urgencia": "alta", "resumen": "El servicio no responde desde hace 2 horas."}
consulta = ConsultaSoporte(**dato_valido)
print(consulta)

dato_invalido = {"categoria": "inventado", "urgencia": "alta", "resumen": "El servicio no responde desde hace 2 horas."}

try:
    ConsultaSoporte(**dato_invalido)
except ValidationError as e:
    print("\nError de validación (esperado):")
    print(e)

**🏠 Practicá (tarea, después de la clase)**: agregá un campo `requiere_escalamiento: bool` al modelo `ConsultaSoporte`. Regenerá el JSON Schema y confirmá que el nuevo campo aparece en `properties`. Probá instanciar el modelo sin ese campo — ¿qué pasa? (pista: Pydantic trata los campos sin default como requeridos).

## 2. Structured outputs

Pedirle al modelo "devolveme JSON" en el prompt **no garantiza** que la salida sea JSON válido, ni que cumpla el schema — el modelo puede agregar texto antes/después, usar comillas simples, olvidar un campo, etc.

OpenAI (igual que AWS Bedrock y algunos otros proveedores) tiene un mecanismo **nativo** que fuerza al modelo a producir una salida que cumple exactamente el schema: en Chat Completions, el campo `response_format` con `type: "json_schema"` y `strict: true`.

In [ ]:
schema_openai = ConsultaSoporte.model_json_schema()
schema_openai["additionalProperties"] = False  # requerido por Structured Outputs en modo strict

texto_libre = "Hace dos días que no puedo entrar a mi cuenta y ya perdí una reunión importante por esto, necesito que lo resuelvan YA."

response = client.chat.completions.create(
    model=model_id,
    messages=[{"role": "user", "content": f"Clasificá esta consulta de soporte: '{texto_libre}'"}],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "consulta_soporte",
            "description": "Clasificación estructurada de una consulta de soporte",
            "strict": True,
            "schema": schema_openai,
        },
    },
)

raw_json = response.choices[0].message.content
print("JSON crudo devuelto por el modelo:", raw_json)

consulta_validada = ConsultaSoporte.model_validate_json(raw_json)
print("\nObjeto Python validado:", consulta_validada)

Notá que igual pasamos el JSON por `ConsultaSoporte.model_validate_json` aunque OpenAI ya garantiza el schema — no es redundante: valida también las reglas que un JSON Schema básico no expresa bien (en este caso ninguna, pero en general sí: por ejemplo relaciones entre campos, o validadores custom de Pydantic).

**🏠 Practicá**: cambiá `texto_libre` por una consulta de facturación de baja urgencia y confirmá que `categoria` y `urgencia` salen correctas sin que se lo digas explícitamente — el modelo infiere la clasificación, no la copia del texto.

## 3. Manejo de respuestas inválidas

No todos los proveedores tienen un mecanismo nativo de structured outputs (Anthropic directo, Google y Ollama no lo tienen igual de simple). Para esos casos, o como red de seguridad general, conviene un patrón universal: **pedir JSON en el prompt, parsear, validar con Pydantic, y si falla, reintentar mostrándole al modelo el error exacto** para que se corrija.

In [ ]:
def generar_estructurado_con_reintentos(llamar_fn, prompt, schema_model, system=None, max_reintentos=3):
    """Patrón universal: sirve con cualquier función llamar_* (Bedrock, OpenAI, Anthropic, Google, Ollama)."""
    instruccion_base = (
        f"{prompt}\n\nRespondé ÚNICAMENTE con un JSON válido que cumpla este schema"
        f"(sin texto adicional, sin markdown):\n{schema_model.model_json_schema()}"
    )
    ultimo_error = None

    for intento in range(max_reintentos):
        instruccion = instruccion_base
        if ultimo_error:
            instruccion += f"\n\nTu respuesta anterior no cumplió el schema. Error: {ultimo_error}\nCorregí y devolvé SOLO el JSON."

        texto = llamar_fn(instruccion, system=system, temperature=0.0)  # temperature baja: queremos formato consistente, no creatividad
        try:
            return schema_model.model_validate_json(texto)
        except (ValidationError, json.JSONDecodeError) as e:
            ultimo_error = str(e)
            print(f"Intento {intento + 1} inválido, reintentando...")

    raise ValueError(f"No se obtuvo una respuesta válida tras {max_reintentos} intentos. Último error: {ultimo_error}")


# Prueba con la función de OpenAI, sin usar el mecanismo nativo (simula un proveedor sin structured outputs)
texto_libre = """
Hace dos días que no puedo entrar a mi cuenta y ya perdí una reunión importante por esto, necesito que lo resuelvan YA. 
"""
resultado = generar_estructurado_con_reintentos(
    llamar_openai,
    f"Clasificá esta consulta de soporte: '{texto_libre}'",
    ConsultaSoporte,
    max_reintentos=1,
)
print(resultado)

**🏠 Practicá**: bajá `max_reintentos` a 1 y modificá el prompt para pedir un campo que no existe en el schema (ej. "incluí también un campo 'telefono'") — el modelo probablemente va a devolver un campo extra o mal formado. Observá el error de Pydantic y confirmá que la función lo reporta con `raise ValueError`, no con un traceback críptico.

## 4. Tool calling

Tool calling es un caso particular de salida estructurada: en vez de pedirle al modelo un objeto de negocio, le das una lista de **herramientas disponibles** (funciones con su schema de argumentos) y el modelo decide *si* necesita alguna y con qué argumentos — vos ejecutás la función real y le devolvés el resultado.

El modelo **nunca ejecuta código** — solo pide "llamá a esta función con estos argumentos". La ejecución es responsabilidad de tu aplicación (esto es clave para seguridad, lo retomamos en la Clase 9 y la Clase 11).

In [ ]:
# La "herramienta" real: una función Python común y corriente
def consultar_estado_ticket(ticket_id: str) -> dict:
    tickets_mock = {"T-001": "En progreso", "T-002": "Resuelto", "T-003": "Esperando al cliente"}
    return {"ticket_id": ticket_id, "estado": tickets_mock.get(ticket_id, "No encontrado")}


# La descripción de esa función que le mandamos al modelo (JSON Schema de sus argumentos)
tools = [{
    "type": "function",
    "function": {
        "name": "consultar_estado_ticket",
        "description": "Consulta el estado actual de un ticket de soporte por su ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string", "description": "ID del ticket, ej. T-001"}
            },
            "required": ["ticket_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}]

In [ ]:
messages = [{"role": "user", "content": "¿Cuál es el estado del ticket T-002?"}]

response = client.chat.completions.create(
    model=model_id,
    messages=messages,
    tools=tools,
)
assistant_msg = response.choices[0].message
messages.append(assistant_msg)

print("Finish reason:", response.choices[0].finish_reason)

if response.choices[0].finish_reason == "tool_calls":
    for tool_call in assistant_msg.tool_calls:
        nombre = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f"El modelo pide llamar a: {nombre}({args})")

        if nombre == "consultar_estado_ticket":
            resultado = consultar_estado_ticket(**args)  # acá ejecutamos NOSOTROS la función real
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(resultado, ensure_ascii=False),
            })

    # Segunda llamada: le devolvemos el resultado de la herramienta para que arme la respuesta final
    response_final = client.chat.completions.create(
        model=model_id,
        messages=messages,
        tools=tools,
    )
    print("\nRespuesta final:", response_final.choices[0].message.content)
    messages.append(response_final.choices[0].message)
else:
    print(assistant_msg.content)

In [ ]:
messages

**🏠 Practicá**: preguntá por un ticket que no existe (ej. "T-999") y confirmá que la respuesta final del modelo refleja el "No encontrado" que devuelve la función — no inventa un estado. Después agregá una segunda herramienta mock (ej. `escalar_ticket(ticket_id, motivo)`) y preguntá algo que dispare esa herramienta en vez de la primera.

## 5. Timeouts, reintentos y backoff

En la Clase 2 armamos `generar_con_reintentos` con backoff exponencial fijo (1s, 2s, 4s...). Dos mejoras para producción:

- **Timeout explícito**: sin él, una llamada colgada puede bloquear tu aplicación indefinidamente.
- **Jitter**: sumar un poco de aleatoriedad al backoff evita que muchos clientes reintenten exactamente al mismo tiempo tras un error masivo (efecto "estampida").

In [ ]:
# Cliente con timeouts explícitos y sin el retry automático del SDK (lo manejamos nosotros)
openai_prod = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    timeout=15.0,       # segundos (connect + read)
    max_retries=0,      # desactivamos el retry interno; lo controlamos abajo
)


def generar_con_reintentos(openai_client, model_id, messages, temperature=0.7, max_tokens=500, max_reintentos=3, **extra):
    for intento in range(max_reintentos):
        try:
            return openai_client.chat.completions.create(
                model=model_id,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                **extra,
            )
        except openai.RateLimitError:
            if intento < max_reintentos - 1:
                espera = (2 ** intento) + random.uniform(0, 1)  # backoff exponencial + jitter
                print(f"Rate limit, reintentando en {espera:.1f}s...")
                time.sleep(espera)
                continue
            raise
        except openai.BadRequestError as err:
            raise ValueError(f"Request inválido: {err}") from err
    raise RuntimeError("Se agotaron los reintentos por rate limit.")


resp = generar_con_reintentos(
    openai_prod,
    model_id,
    messages=[{"role": "user", "content": "Decí 'ok'."}],
)
print(resp.choices[0].message.content)

**🏠 Practicá**: bajá `timeout` a `0.01` segundos y volvé a correr la celda anterior — debería fallar con un timeout. Envolvé la llamada en un `try/except` que capture ese error específico (`openai.APITimeoutError`) y decidí si tiene sentido reintentar (sí: un timeout es, en general, reintentable).

## 6. Observabilidad básica: tokens, costo y latencia

Sin esto, no hay forma de responder "¿cuánto nos está costando esto?" ni de detectar una llamada anormalmente lenta. Registrarlo desde el día 1 es mucho más barato que agregarlo después.

In [ ]:
# Valores de ejemplo — el precio real cambia con el tiempo, verificar en la consola de OpenAI antes de usar en producción
# Referencia orientativa para gpt-4o (USD por millón de tokens):
PRECIO_POR_MILLON_USD = {"input": 2.5, "output": 10.0}

def estimar_costo_usd(usage) -> float:
    costo_input = usage.prompt_tokens / 1_000_000 * PRECIO_POR_MILLON_USD["input"]
    costo_output = usage.completion_tokens / 1_000_000 * PRECIO_POR_MILLON_USD["output"]
    return costo_input + costo_output


registro_llamadas = []  # en una app real, esto va a un log estructurado o una base de datos (Clase 11)

def llamar_con_registro(openai_client, model_id, messages, **kwargs):
    t0 = time.time()
    resp = generar_con_reintentos(openai_client, model_id, messages, **kwargs)
    latencia_ms = int((time.time() - t0) * 1000)
    registro_llamadas.append({
        "input_tokens": resp.usage.prompt_tokens,
        "output_tokens": resp.usage.completion_tokens,
        "costo_usd": estimar_costo_usd(resp.usage),
        "latencia_ms": latencia_ms,
    })
    return resp

resp = llamar_con_registro(
    openai_prod,
    model_id,
    messages=[{"role": "user", "content": "Explicá qué es un embedding en 1 oración."}],
)
print(resp.choices[0].message.content)
print("\nRegistro:", registro_llamadas[-1])

**🏠 Practicá**: hacé 5 llamadas distintas con `llamar_con_registro` y al final calculá el costo total y la latencia promedio a partir de `registro_llamadas` (con una simple suma/promedio sobre la lista de dicts). Esto es literalmente lo que vas a necesitar para el informe de costos del TP Final (Clase 12).

## 7. Streaming

Para respuestas largas, esperar a que termine toda la generación antes de mostrar nada da una sensación de latencia alta. El streaming muestra el texto a medida que se genera, token a token.

In [ ]:
start = time.time()
stream = client.chat.completions.create(
    model=model_id,
    messages=[{"role": "user", "content": "Listá 5 buenas prácticas de seguridad para agentes de IA, con una oración de explicación cada una."}],
    stream=True,
    stream_options={"include_usage": True},
)

usage = None
for event in stream:
    if event.choices:
        delta = event.choices[0].delta.content
        if delta:
            print(f"{time.time() - start:.3f}", end=" ")
            print(delta, end="", flush=True)
    if event.usage is not None:
        usage = event.usage

print(f"\n\n[tokens: prompt={usage.prompt_tokens if usage else '?'}, completion={usage.completion_tokens if usage else '?'}]")
print(f"total: {time.time() - start:.3f}s")

**🏠 Practicá**: medí con `time.time()` cuánto tarda en llegar el **primer** chunk de texto (time to first token) vs. cuánto tarda la respuesta completa con `chat.completions.create` normal (sin streaming) para el mismo prompt. Esa diferencia es la que justifica usar streaming en una UI interactiva.

## 8. Abstracción de proveedores

Hasta acá todo el notebook (y el `LLMClient` de la Clase 2) llama directo a OpenAI, con un `if/elif` por proveedor en el mejor de los casos. Eso no escala: cada vez que agregás lógica nueva (reintentos, structured outputs, registro de costos) hay que replicarla en cada rama.

La solución: una **interfaz común** (clase abstracta) que cada proveedor implementa. El resto de la aplicación programa contra la interfaz, no contra un SDK específico.

In [ ]:
from abc import ABC, abstractmethod


class LLMProvider(ABC):
    """Interfaz común para cualquier proveedor de modelos de lenguaje."""

    @abstractmethod
    def generate(self, prompt: str, system: str = None, temperature: float = 0.7, max_tokens: int = 500) -> str:
        """Devuelve el texto generado por el modelo."""
        ...

    def generate_structured(self, prompt: str, schema_model, system: str = None):
        """Implementación por defecto: sirve para CUALQUIER proveedor, aunque no tenga
        soporte nativo de structured outputs. Un proveedor que sí lo tenga (como OpenAI)
        puede sobreescribir este método con una versión más eficiente (ver OpenAIProvider)."""
        return generar_estructurado_con_reintentos(self.generate, prompt, schema_model, system=system)

In [ ]:
class OpenAIProvider(LLMProvider):
    def __init__(self):
        self.client = OpenAI(
            api_key=os.environ["OPENAI_API_KEY"],
            timeout=15.0,
            max_retries=0,
        )
        self.model_id = os.environ.get("OPENAI_MODEL_ID", "gpt-4o")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        resp = generar_con_reintentos(
            self.client, self.model_id, messages,
            temperature=temperature, max_tokens=max_tokens,
        )
        return resp.choices[0].message.content

    def generate_structured(self, prompt, schema_model, system=None):
        # Override: usa el mecanismo nativo de OpenAI (sección 2) en vez del reintento genérico
        schema = schema_model.model_json_schema()
        schema["additionalProperties"] = False
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        resp = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": schema_model.__name__,
                    "description": schema_model.__doc__ or "",
                    "strict": True,
                    "schema": schema,
                },
            },
        )
        raw = resp.choices[0].message.content
        return schema_model.model_validate_json(raw)

Los demás proveedores quedan como ejercicio — solo necesitan implementar `generate()` (ya lo tenés resuelto de la Clase 2, sección "Tu proveedor"); `generate_structured()` viene gratis de la clase base. Si tu proveedor tiene structured outputs nativos (como Bedrock), podés sobreescribir `generate_structured` igual que hicimos con OpenAI.

In [ ]:
class BedrockProvider(LLMProvider):
    def __init__(self):
        import boto3
        from botocore.config import Config
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=os.environ.get("AWS_REGION", "us-east-1"),
            config=Config(connect_timeout=5, read_timeout=15, retries={"max_attempts": 0}),
        )
        self.model_id = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_bedrock (Clase 2, sección 2)
        raise NotImplementedError


class AnthropicProvider(LLMProvider):
    def __init__(self):
        import anthropic
        self.client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        self.model_id = os.environ.get("ANTHROPIC_MODEL_ID", "claude-sonnet-4-6")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_anthropic (Clase 2, sección 2)
        raise NotImplementedError


class GoogleProvider(LLMProvider):
    def __init__(self):
        from google import genai
        self.client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
        self.model_id = os.environ.get("GOOGLE_MODEL_ID", "gemini-2.0-flash")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_google (Clase 2, sección 2)
        raise NotImplementedError


class OllamaProvider(LLMProvider):
    def __init__(self):
        self.model_id = os.environ.get("OLLAMA_MODEL_ID", "llama3.1")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_ollama (Clase 2, sección 2)
        raise NotImplementedError


def get_provider() -> LLMProvider:
    """Factory: devuelve la instancia del proveedor configurado en LLM_PROVIDER."""
    proveedores = {
        "bedrock": BedrockProvider,
        "anthropic": AnthropicProvider,
        "openai": OpenAIProvider,
        "google": GoogleProvider,
        "ollama": OllamaProvider,
    }
    nombre = os.environ.get("LLM_PROVIDER", "openai")
    if nombre not in proveedores:
        raise ValueError(f"Proveedor desconocido: {nombre}")
    return proveedores[nombre]()


# Prueba: esto funciona hoy con OpenAI. Cuando completes los TODO de arriba, va a funcionar
# igual (sin cambiar esta celda) con cualquier otro proveedor que configures en LLM_PROVIDER.
llm = get_provider()
print(llm.generate("Decí 'funciona' y nada más."))

**🏠 Practicá**: completá al menos un `TODO` (el de tu proveedor real, si no es OpenAI) y confirmá que `get_provider().generate(...)` funciona sin tocar ninguna otra celda del notebook. Después probá `get_provider().generate_structured("Clasificá esta consulta: 'no me llegó la factura'", ConsultaSoporte)` — con OpenAI usa el mecanismo nativo (sección 2); con cualquier otro proveedor, usa el reintento genérico (sección 3) automáticamente, sin que hayas escrito nada extra para eso.

## 9. Laboratorio: servicio de extracción estructurada

**🏠 Tarea de la semana — entregar/completar antes de la Clase 4.** El docente presenta la consigna y muestra el patrón en vivo, pero la resolución es trabajo individual asincrónico.

**Checkpoint del proyecto integrador (Clase 3)**: el `LLMClient` de la Clase 2, ahora detrás de la interfaz `LLMProvider`, más un servicio que transforma texto libre en objetos validados.

### Consigna

1. Terminá de completar los `TODO` de `BedrockProvider` / `AnthropicProvider` / `GoogleProvider` / `OllamaProvider` — al menos el de **tu** proveedor (si no usás OpenAI).
2. Definí un modelo Pydantic propio (no `ConsultaSoporte`) relevante para el dominio del TP integrador — por ejemplo, una clasificación de la intención de una pregunta sobre la documentación técnica del corpus (`tipo_pregunta`, `requiere_busqueda_en_documentacion: bool`, `resumen`).
3. Escribí una función `clasificar(texto: str) -> TuModelo` que use `get_provider().generate_structured(...)` para resolverlo.
4. Probala con al menos 3 textos de entrada distintos, incluyendo uno ambiguo o mal escrito, y confirmá que **siempre** devuelve un objeto válido o levanta una excepción clara (nunca un `dict` sin validar ni un texto crudo).

### Criterio de éxito

- El servicio nunca deja pasar un objeto que no cumple tu schema Pydantic — o devuelve el objeto validado, o falla explícitamente.
- Funciona con el proveedor configurado en tu `.env`, sin credenciales hardcodeadas.
- Podés explicar en una frase la diferencia entre usar `generate_structured` con OpenAI (nativo) y con cualquier otro proveedor (reintento + prompt) — sin que el código que llama a `clasificar()` necesite saberlo.

In [ ]:
class MiModelo(BaseModel):
    # TODO: definí los campos relevantes para tu caso de uso del TP
    pass


def clasificar(texto: str) -> "MiModelo":
    # TODO: usar get_provider().generate_structured(...)
    raise NotImplementedError


# Prueba (va a fallar hasta que completes los TODOs):
# print(clasificar("¿Cómo configuro las credenciales de OpenAI?"))

## Cierre

- Hoy el `LLMClient` de la Clase 2 se convirtió en algo bastante más sólido: interfaz común (`LLMProvider`), salidas estructuradas y validadas, tool calling, timeouts, reintentos con jitter, registro de costos y streaming.
- Esto **es** el primer entregable serio del TP integrador — de acá en adelante (RAG, agentes, despliegue) todo se construye sobre esta base, no se vuelve a escribir.
- Antes de la próxima clase: dejar el laboratorio de la sección 9 funcionando con tu proveedor.
- Próxima clase: embeddings e ingesta documental — empezamos a preparar el corpus para la recuperación semántica.